# Multiple customer classes in a model

In this example, the model has three customer classes. The customer class determines the arrival distribution and service distribution.

The JSON for this built-in example can be loaded using `json2ciw.datasets.load_three_classes_model`.

## Imports

In [1]:
import json

import ciw
from rich import print

from json2ciw.datasets import load_three_classes_model
from json2ciw.engine import CiwConverter, multiple_replications
from json2ciw.results import summarise_results, summarise_results_by_class, tidy_to_wide_format, tidy_to_wide_format_by_class
from json2ciw.schema import ProcessModel

## Load JSON

In [2]:
json_network = load_three_classes_model()
print(json.dumps(json_network, indent=2))

{
  "name": "Simple Critical Care Model",
  "description": "Three customer classes arriving to a shared critical care activity.",
  "customer_classes": [
    {
      "name": "emergency_department",
      "label": "Emergency Department"
    },
    {
      "name": "surgery",
      "label": "Surgery"
    },
    {
      "name": "elective",
      "label": "Elective"
    }
  ],
  "activities": [
    {
      "name": "Critical care",
      "type": "activity",
      "resource": {
        "name": "Critical care beds",
        "capacity": 10
      },
      "arrival_distribution": {
        "by_class": {
          "emergency_department": {
            "type": "exponential",
            "parameters": {
              "mean": 1.0
            }
          },
          "surgery": {
            "type": "exponential",
            "parameters": {
              "mean": 3.0
            }
          },
          "elective": {
            "type": "exponential",
            "parameters": {
              "mean": 6.0
            }
          }
        }
      },
      "service_distribution": {
        "by_class": {
          "emergency_department": {
            "type": "gamma",
            "parameters": {
              "shape": 3.0,
              "scale": 2.0
            }
          },
          "surgery": {
            "type": "gamma",
            "parameters": {
              "shape": 2.5,
              "scale": 1.8
            }
          },
          "elective": {
            "type": "gamma",
            "parameters": {
              "shape": 2.0,
              "scale": 1.5
            }
          }
        }
      }
    },
    {
      "name": "Discharge",
      "type": "activity",
      "resource": {
        "name": "Discharge staff",
        "capacity": 2
      },
      "service_distribution": {
        "type": "deterministic",
        "parameters": {
          "value": 0.5
        }
      }
    }
  ],
  "transitions": [
    {
      "from": "Critical care",
      "to": "Discharge",
      "probability": 1.0
    },
    {
      "from": "Discharge",
      "to": "Exit",
      "probability": 1.0
    }
  ]
}

## Validate with `ProcessModel`

In [3]:
model_instance = ProcessModel(**json_network)

In [4]:
print(model_instance)

ProcessModel(
    name='Simple Critical Care Model',
    description='Three customer classes arriving to a shared critical care activity.',
    customer_classes=[
        CustomerClass(name='emergency_department', label='Emergency Department'),
        CustomerClass(name='surgery', label='Surgery'),
        CustomerClass(name='elective', label='Elective')
    ],
    activities=[
        Activity(
            name='Critical care',
            type='activity',
            resource=Resource(name='Critical care beds', capacity=10),
            service_distribution=ClassDistributionMap(
                by_class={
                    'emergency_department': Distribution(type='gamma', parameters={'shape': 3.0, 'scale': 2.0}),
                    'surgery': Distribution(type='gamma', parameters={'shape': 2.5, 'scale': 1.8}),
                    'elective': Distribution(type='gamma', parameters={'shape': 2.0, 'scale': 1.5})
                }
            ),
            arrival_distribution=ClassDistributionMap(
                by_class={
                    'emergency_department': Distribution(type='exponential', parameters={'mean': 1.0}),
                    'surgery': Distribution(type='exponential', parameters={'mean': 3.0}),
                    'elective': Distribution(type='exponential', parameters={'mean': 6.0})
                }
            ),
            renege_distribution=None
        ),
        Activity(
            name='Discharge',
            type='activity',
            resource=Resource(name='Discharge staff', capacity=2),
            service_distribution=Distribution(type='deterministic', parameters={'value': 0.5}),
            arrival_distribution=None,
            renege_distribution=None
        )
    ],
    transitions=[
        Transition(source='Critical care', target='Discharge', probability=1.0),
        Transition(source='Discharge', target='Exit', probability=1.0)
    ]
)

In [5]:
model_instance.display_diagram(include_resources=False, show_class_arrivals=True)

```mermaid 
graph TD
    Arrivals_Critical_care_emergency_department("Emergency Department</br>Time between arrivals<br/>Exponential(mean=1.0)")
    Arrivals_Critical_care_surgery("Surgery</br>Time between arrivals<br/>Exponential(mean=3.0)")
    Arrivals_Critical_care_elective("Elective</br>Time between arrivals<br/>Exponential(mean=6.0)")
    Critical_care["Critical care</br>Class-specific service distributions (n=3)"]
    Discharge["Discharge</br>Deterministic(0.5)"]
    Exit(["Exit"])

    Arrivals_Critical_care_emergency_department --> Critical_care
    Arrivals_Critical_care_surgery --> Critical_care
    Arrivals_Critical_care_elective --> Critical_care
    Critical_care --> Discharge
    Discharge --> Exit 
```

In [6]:
model_instance.save_diagram("example7.mmd", include_resources=False)

In [7]:
model_instance.get_distributions_df()

,Activity,Phase,Customer Class,Customer Class Label,Distribution Type,Parameters
0,Critical care,Arrival,emergency_department,Emergency Department,Exponential,mean=1.0
1,Critical care,Arrival,surgery,Surgery,Exponential,mean=3.0
2,Critical care,Arrival,elective,Elective,Exponential,mean=6.0
3,Critical care,Service,emergency_department,Emergency Department,Gamma,"shape=3.0, scale=2.0"
4,Critical care,Service,surgery,Surgery,Gamma,"shape=2.5, scale=1.8"
5,Critical care,Service,elective,Elective,Gamma,"shape=2.0, scale=1.5"
6,Discharge,Service,All,All,Deterministic,value=0.5


In [8]:
model_instance.get_routing_matrix_df()

,Critical care,Discharge,Exit
Source Activity,,,
Critical care,0.0,1.0,0.0
Discharge,0.0,0.0,1.0


In [9]:
model_instance.get_resources_df()

,Resource,Activity,Count
0,Critical care beds,Critical care,10
1,Discharge staff,Discharge,2


## Convert to `ciw` parameters

In [10]:
adapter = CiwConverter(model_instance)
network_params = adapter.generate_params()
print(network_params)

{
    'number_of_servers': [10, 2],
    'arrival_distributions': {
        'emergency_department': [Exponential(rate=1.0), None],
        'surgery': [Exponential(rate=0.3333333333333333), None],
        'elective': [Exponential(rate=0.16666666666666666), None]
    },
    'service_distributions': {
        'emergency_department': [Gamma(shape=3.0, scale=2.0), Deterministic(value=0.5)],
        'surgery': [Gamma(shape=2.5, scale=1.8), Deterministic(value=0.5)],
        'elective': [Gamma(shape=2.0, scale=1.5), Deterministic(value=0.5)]
    },
    'routing': {
        'emergency_department': [[0.0, 1.0], [0.0, 0.0]],
        'surgery': [[0.0, 1.0], [0.0, 0.0]],
        'elective': [[0.0, 1.0], [0.0, 0.0]]
    }
}

## Build and run the `ciw` model

In [11]:
network = ciw.create_network(**network_params)
sim = ciw.Simulation(network)
sim.simulate_until_max_time(50)
print("Quick simulation run worked!")

Quick simulation run worked!

## Run the model for multiple replications

In [12]:
df_reps = multiple_replications(
    network,
    model_instance,
    num_reps=5,
    runtime=2880,
    warmup=1440,
    n_jobs=-1,
)

df_reps.head()

,rep,node_id,activity_name,resource_name,resource_capacity,measure_scope,customer_class,n_service,mean_wait,mean_service,mean_Lq,utilisation
0,0,1,Critical care,Critical care beds,10,overall,All,2105,0.741356,5.357066,1.083718,80.741572
1,0,1,Critical care,Critical care beds,10,customer_class,emergency_department,1441,0.733161,6.002965,0.733670,NaN
2,0,1,Critical care,Critical care beds,10,customer_class,surgery,431,0.759284,4.532642,0.227258,NaN
3,0,1,Critical care,Critical care beds,10,customer_class,elective,233,0.758875,2.887480,0.122790,NaN
4,0,2,Discharge,Discharge staff,2,overall,All,2110,0.035996,0.500000,0.052745,37.126736


## Convert to wide format

In [13]:
# overall results
wide = tidy_to_wide_format(df_reps)
wide.head()

,mean_Lq [Critical care],mean_Lq [Discharge],mean_service [Critical care],mean_service [Discharge],mean_wait [Critical care],mean_wait [Discharge],n_service [Critical care],n_service [Discharge],utilisation [Critical care],utilisation [Discharge]
rep,,,,,,,,,,
0,1.083718,0.052745,5.357066,0.5,0.741356,0.035996,2105,2110,80.741572,37.126736
1,1.335788,0.052716,5.354101,0.5,0.908614,0.035706,2117,2126,81.561898,37.986111
2,0.640339,0.053848,5.255430,0.5,0.443738,0.037155,2078,2087,78.428494,37.196181
3,2.022202,0.053509,5.400027,0.5,1.336993,0.035265,2178,2185,79.515973,37.256944
4,1.637887,0.055620,5.361820,0.5,1.106784,0.037461,2131,2138,79.130466,36.774145


In [14]:
# results by class = emergency_department
wide_by_class = tidy_to_wide_format_by_class(df_reps, customer_class="emergency_department")
wide_by_class.head()

,mean_Lq [Critical care],mean_Lq [Discharge],mean_service [Critical care],mean_service [Discharge],mean_wait [Critical care],mean_wait [Discharge],n_service [Critical care],n_service [Discharge]
rep,,,,,,,,
0,0.733670,0.036585,6.002965,0.5,0.733161,0.036509,1441,1443
1,0.888996,0.035356,6.076416,0.5,0.913092,0.036263,1402,1404
2,0.405326,0.035419,5.962794,0.5,0.428853,0.037311,1361,1367
3,1.366238,0.035579,6.060122,0.5,1.363397,0.035382,1443,1448
4,1.147095,0.039265,5.962894,0.5,1.142336,0.038941,1446,1452


In [18]:
# results by class = surgery
wide_by_class = tidy_to_wide_format_by_class(df_reps, customer_class="surgery")
wide_by_class.head()

,mean_Lq [Critical care],mean_Lq [Discharge],mean_service [Critical care],mean_service [Discharge],mean_wait [Critical care],mean_wait [Discharge],n_service [Critical care],n_service [Discharge]
rep,,,,,,,,
0,0.227258,0.011632,4.532642,0.5,0.759284,0.038596,431,434
1,0.311373,0.010849,4.409366,0.5,0.938028,0.032344,478,483
2,0.155696,0.012971,4.391801,0.5,0.477027,0.039574,470,472
3,0.442615,0.012719,4.635279,0.5,1.298097,0.037150,491,493
4,0.337345,0.010591,4.574393,0.5,1.084324,0.034043,448,448


## Summarise results

In [15]:
df_reps.head(2)

,rep,node_id,activity_name,resource_name,resource_capacity,measure_scope,customer_class,n_service,mean_wait,mean_service,mean_Lq,utilisation
0,0,1,Critical care,Critical care beds,10,overall,All,2105,0.741356,5.357066,1.083718,80.741572
1,0,1,Critical care,Critical care beds,10,customer_class,emergency_department,1441,0.733161,6.002965,0.733670,NaN


In [16]:
summary = summarise_results(df_reps)
summary.round(1)

activity,Metric,Critical care (Critical care beds),Discharge (Discharge staff)
0,Mean completed services,2121.8,2129.2
1,Mean waiting time,0.9,0.0
2,Mean service time,5.3,0.5
3,Mean utilisation,79.9,37.3
4,Mean queue length,1.3,0.1


In [17]:
summary_class = summarise_results_by_class(df_reps)
summary_class.round(1)

,Activity,Customer Class,Mean completed services,Mean waiting time,Mean service time,Mean queue length
0,Critical care,elective,239.6,0.8,3.0,0.1
1,Critical care,emergency_department,1418.6,0.9,6.0,0.9
2,Critical care,surgery,463.6,0.9,4.5,0.3
3,Discharge,elective,240.4,0.0,0.5,0.0
4,Discharge,emergency_department,1422.8,0.0,0.5,0.0
5,Discharge,surgery,466.0,0.0,0.5,0.0
